# Entraîner et piloter l'affinage du service client

Le fil rouge de la formation : le service client de Biclou & Cie, dont le modèle doit trier chaque message (catégorie, priorité) et y répondre selon la charte, en un objet JSON. Le TP précédent a configuré QLoRA ; celui-ci entraîne pour de bon Qwen3-1.7B sur le jeu du TP de données, en suivant les courbes pendant l'entraînement, compare deux runs, mesure le modèle adapté face au modèle de base, puis enregistre l'adaptateur avec ce qu'il faut pour le retrouver.

1. Un premier entraînement, suivi en direct
2. Lire les courbes, lancer un second run
3. Le modèle adapté face au modèle de base
4. Enregistrer et publier l'adaptateur

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*, sur lequel il tient en une heure environ, dont la plus grande part pour les deux entraînements. Sans GPU, il tourne quand même, en plus petit : Qwen3-0.6B en float32 au lieu de Qwen3-1.7B en 4 bits, entraîné sur 2 exemples au lieu de 1 190 (le code tourne, mais les courbes ne disent rien), et l'exercice 3 mesure, sur 4 tickets de test au lieu de 150, un adaptateur de Qwen3-0.6B que ce notebook a entraîné sur GPU.

## Données et modèle

`support-sft-train.jsonl` et `support-sft-validation.jsonl` : le jeu d'entraînement (1 190 exemples) et le jeu de validation (133) que produit le TP de données, celui de sa version `v1.0`, au format conversationnel *prompt / completion* de `trl`. `test.jsonl` : les 150 tickets de test validés par l'équipe, ceux du TP de mesure. `support-sft-lora-qwen3-0.6b/` : l'adaptateur de Qwen3-0.6B que mesure l'exercice 3 quand le notebook tourne sans GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/support-sft-train.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/support-sft-validation.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/test.jsonl
!wget -q -P support-sft-lora-qwen3-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/support-sft-lora-qwen3-0.6b/adapter_model.safetensors
!wget -q -P support-sft-lora-qwen3-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/support-sft-lora-qwen3-0.6b/adapter_config.json
%pip install -q peft trl bitsandbytes
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

In [ ]:
import gc
import json
import os
import re
import time

import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

GPU = torch.cuda.is_available()
# bfloat16 à partir des GPU Ampere (A100, L4, H100) ; le T4 calcule en float16.
BF16 = GPU and torch.cuda.get_device_capability()[0] >= 8
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = (torch.bfloat16 if BF16 else torch.float16) if GPU else torch.float32
DEVICE = "cuda" if GPU else "cpu"
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
# Exemples par pas et pas accumulés par mise à jour : 16 exemples sur GPU, 1 sans.
LOT = 4 if GPU else 1
ACCUMULATION = 4 if GPU else 1
print(MODELE, "sur GPU" if GPU else "sans GPU")

tokenizer = AutoTokenizer.from_pretrained(MODELE)
entrainement = load_dataset("json", data_files="support-sft-train.jsonl", split="train")
validation = load_dataset(
  "json", data_files="support-sft-validation.jsonl", split="train"
)
test = pd.read_json("test.jsonl", lines=True)
# Sans GPU : 2 exemples d'entraînement, 1 de validation et 4 tickets de test.
if not GPU:
  entrainement = entrainement.select(range(2))
  validation = validation.select(range(1))
test_part = test[: None if GPU else 4]
print(len(entrainement), "exemples d'entraînement,", len(validation), "de validation")


def memoire_go(modele) -> float:
  """La mémoire qu'occupe le modèle, en Go : celle du GPU, sinon celle de ses poids."""
  if GPU:
    return torch.cuda.memory_allocated() / 1e9
  return modele.get_memory_footprint() / 1e9


def pic_go() -> float:
  """Le pic de mémoire GPU depuis le dernier libere(), en Go (0 sans GPU)."""
  return torch.cuda.max_memory_allocated() / 1e9 if GPU else 0.0


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`, et remet le pic à zéro."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def taille_mo(dossier: str) -> float:
  """La taille des fichiers d'un dossier, en Mo."""
  return (
    sum(os.path.getsize(os.path.join(dossier, f)) for f in os.listdir(dossier)) / 1e6
  )

## La mesure

Celle du TP de mesure, telle quelle : `CONSIGNES`, la consigne courte, que reçoit chaque exemple d'entraînement ; `evalue(sorties)`, qui compare des réponses à celles de `test_part` (`json_valide`, `categorie`, `priorite`, `charte`, `mots`) ; `genere_tout(conversations)`, qui génère par lots de 8, sans échantillonnage, la réponse de `modele` à chaque conversation.

In [ ]:
CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
CONSIGNES = f"""Tu es l'assistant du service client de Biclou & Cie, magasins de vélos.
Pour le message du client, réponds uniquement par un objet JSON avec trois clés :
- categorie : une valeur parmi {", ".join(CATEGORIES)} ;
- priorite : basse, normale ou haute ;
- reponse : la réponse à envoyer au client."""
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)


def lit_json(texte: str) -> dict:
  """L'objet JSON d'une réponse du modèle, ou {} s'il n'est pas valide."""
  texte = texte.strip().removeprefix("```json").removesuffix("```").strip()
  try:
    objet = json.loads(texte)
  except json.JSONDecodeError:
    return {}
  return objet if isinstance(objet, dict) else {}


def respecte_charte(reponse: str) -> bool:
  reponse = reponse.strip()
  return (
    reponse.startswith("Bonjour")
    and reponse.endswith(SIGNATURE)
    and not TUTOIEMENT.search(reponse)
    and 40 <= len(reponse.split()) <= 120
  )


def evalue(sorties: list[str]) -> pd.Series:
  objets = [lit_json(s) for s in sorties]
  reponses = [str(o.get("reponse", "")) for o in objets]
  return (
    pd.Series(
      {
        "json_valide": sum(
          {"categorie", "priorite", "reponse"} <= o.keys() for o in objets
        ),
        "categorie": sum(
          o.get("categorie") == c for o, c in zip(objets, test_part["categorie"])
        ),
        "priorite": sum(
          o.get("priorite") == p for o, p in zip(objets, test_part["priorite"])
        ),
        "charte": sum(respecte_charte(r) for r in reponses),
        "mots": sum(len(r.split()) for r in reponses),
      }
    )
    .div(len(sorties))
    .round(2)
  )


def genere_tout(conversations: list[list[dict]], taille_lot: int = 8) -> list[str]:
  textes = []
  for debut in range(0, len(conversations), taille_lot):
    prompts = [
      tokenizer.apply_chat_template(
        c, tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for c in conversations[debut : debut + taille_lot]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=384,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    textes += tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes

## 1. Un premier entraînement, suivi en direct

La cellule ci-dessous charge le modèle de base, `base`, en NF4 avec double quantification, et reprend dans `qlora` et `REFERENCE` la configuration de l'exercice 3 du TP précédent. Écrivez `prepare(nom, **changements)`, qui renvoie un `SFTTrainer` prêt à entraîner l'adaptateur `qlora` sur `base` pendant une époque d'`entrainement`, avec les réglages de `REFERENCE` (que `changements` remplacent), et de quoi suivre l'entraînement :

- les journaux dans TensorBoard, sous `sft/<nom>`, avec la perte d'entraînement à chaque pas ;
- une évaluation sur `validation` à chaque dixième de l'entraînement, et un checkpoint à chacune ;
- deux checkpoints gardés au plus, dont toujours le meilleur selon la perte de validation, rechargé à la fin.

Lancez TensorBoard (la deuxième cellule ci-dessous), puis préparez le premier run, `prepare("run-1")`. Avant de l'entraîner, regardez le premier exemple tel que le *trainer* l'a préparé (`trainer.train_dataset[0]`) : combien de ses tokens comptent dans la perte, et lesquels ? Sur un jeu *prompt / completion*, `trl` ne calcule la perte que sur la complétion (`completion_only_loss`, actif par défaut sur ce format) : le modèle apprend à répondre, pas à écrire les messages des clients. Le plus long exemple tient-il dans `max_length` ?

Puis entraînez, et suivez dans TensorBoard `train/loss`, `eval/loss` et `train/learning_rate` (il relit les journaux toutes les 30 secondes ; ⟳, en haut à droite, sans attendre). À quel pas la perte d'entraînement cesse-t-elle de chuter ? La perte de validation suit-elle ? Notez la durée, le pic de mémoire, le meilleur checkpoint (`trainer.state.best_model_checkpoint`) et sa perte (`best_metric`).

Sans GPU, `entrainement` n'a que 2 exemples et `validation` 1 : deux pas d'un exemple chacun, pour voir la boucle tourner.

**Documentation utile**

- [`SFTConfig`](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig)
- [La perte sur la complétion seule](https://huggingface.co/docs/trl/sft_trainer#train-on-completion-only)
- [`TrainingArguments`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)
- [TensorBoard dans un notebook](https://www.tensorflow.org/tensorboard/tensorboard_in_notebooks)
- [`TrainerState`](https://huggingface.co/docs/transformers/main_classes/callback#transformers.TrainerState)

In [ ]:
NF4 = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=DTYPE,
)
# Sans GPU, en float32 : bitsandbytes est lent sur processeur.
base = AutoModelForCausalLM.from_pretrained(
  MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=NF4 if GPU else None
)
print(f"Modèle de base : {memoire_go(base):.2f} Go")

# La configuration de l'exercice 3 du TP précédent.
qlora = LoraConfig(
  r=16,
  lora_alpha=32,
  lora_dropout=0.05,
  target_modules="all-linear",
  task_type="CAUSAL_LM",
)
REFERENCE = dict(
  per_device_train_batch_size=LOT,
  per_device_eval_batch_size=LOT,
  gradient_accumulation_steps=ACCUMULATION,
  learning_rate=2e-4,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  optim="paged_adamw_8bit" if GPU else "adamw_torch",
  # Économise la mémoire du GPU au prix d'un passage avant de plus ; inutile sans GPU.
  gradient_checkpointing=GPU,
  bf16=BF16,
  fp16=GPU and not BF16,
  max_length=1024,
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir sft

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Dans les réglages : `num_train_epochs=1` ; `report_to="tensorboard"` ; `logging_steps=1` ; `eval_strategy="steps"`, `eval_steps=0.1` ; `save_strategy="steps"`, `save_steps=0.1` ; `save_total_limit=2` ; `load_best_model_at_end=True`, `metric_for_best_model="eval_loss"`
- Un nombre sous 1, pour `eval_steps` ou `save_steps`, est une part des pas de l'entraînement : 0,1, un dixième ; pour recharger le meilleur checkpoint à la fin, `save_steps` doit être un multiple d'`eval_steps`
- Les réglages du suivi dans un dictionnaire, `suivi = dict(num_train_epochs=1, ...)`, puis `SFTConfig(output_dir=f"sft/{nom}", **(REFERENCE | suivi | changements))` : `|` réunit deux dictionnaires, la clé de droite remplaçant la même clé à gauche
- TensorBoard lit les journaux sous `sft/<nom>/runs/` : `%tensorboard --logdir sft` montre tous les runs
- `SFTTrainer(model=base, args=config, train_dataset=entrainement, eval_dataset=validation, processing_class=tokenizer, peft_config=qlora)`
- Un exemple préparé a des `input_ids` et des `labels` de même longueur : `-100` dans `labels` exclut le token de la perte ; `tokenizer.decode(tokens)` les remet en texte
- Après `trainer.train()` : `pic_go()`, `trainer.state.best_model_checkpoint`, `trainer.state.best_metric`

```python
def prepare(nom: str, **changements) -> SFTTrainer:
  suivi = dict(
    num_train_epochs=1,
    report_to=...,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy=...,
    save_steps=...,
    save_total_limit=2,
    load_best_model_at_end=...,
    metric_for_best_model=...,
  )
  config = SFTConfig(output_dir=f"sft/{nom}", **(REFERENCE | suivi | changements))
  return SFTTrainer(
    model=base,
    args=config,
    train_dataset=...,
    eval_dataset=...,
    processing_class=tokenizer,
    peft_config=qlora,
  )


trainer = prepare("run-1")
exemple = trainer.train_dataset[0]
comptes = [t for t, l in zip(exemple["input_ids"], exemple["labels"]) if l != -100]
print(f"{len(comptes)} tokens sur {len(exemple['input_ids'])} comptent dans la perte :")
print(tokenizer.decode(comptes))
longueurs = [len(ids) for ids in trainer.train_dataset["input_ids"]]
print(f"Le plus long exemple : {max(longueurs)} tokens")

debut = time.time()
trainer.train()
print(f"{(time.time() - debut) / 60:.1f} min, pic {pic_go():.1f} Go")
print("Meilleur checkpoint :", ..., f"perte de validation {...:.3f}")
```

### Solution

In [ ]:
def prepare(nom: str, **changements) -> SFTTrainer:
  suivi = dict(
    num_train_epochs=1,
    report_to="tensorboard",
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy="steps",
    save_steps=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
  )
  config = SFTConfig(output_dir=f"sft/{nom}", **(REFERENCE | suivi | changements))
  return SFTTrainer(
    model=base,
    args=config,
    train_dataset=entrainement,
    eval_dataset=validation,
    processing_class=tokenizer,
    peft_config=qlora,
  )


trainer = prepare("run-1")
exemple = trainer.train_dataset[0]
comptes = [t for t, l in zip(exemple["input_ids"], exemple["labels"]) if l != -100]
print(f"{len(comptes)} tokens sur {len(exemple['input_ids'])} comptent dans la perte :")
print(tokenizer.decode(comptes))
longueurs = [len(ids) for ids in trainer.train_dataset["input_ids"]]
print(f"Le plus long exemple : {max(longueurs)} tokens")

debut = time.time()
trainer.train()
print(f"{(time.time() - debut) / 60:.1f} min, pic {pic_go():.1f} Go")
print(
  "Meilleur checkpoint :",
  trainer.state.best_model_checkpoint,
  f"perte de validation {trainer.state.best_metric:.3f}",
)

# La complétion seule compte : le bloc de réflexion vide que le modèle de conversation
# de Qwen3 écrit sans réflexion (<think></think>), le JSON, puis la fin du tour. Les
# exemples font environ 300 tokens, 370 au plus : max_length=1024 n'en coupe aucun.

## 2. Lire les courbes, lancer un second run

La cellule ci-dessous garde l'état du premier run, `etat_1` (`trainer.state` : son historique, son meilleur checkpoint), et retire son adaptateur du modèle de base, pour le run suivant.

Relisez les courbes du premier run : la perte de validation baisse-t-elle encore à la fin ? Entraîner plus longtemps aiderait-il ? Lancez un second run, `run-2`, qui ne change qu'un réglage : deux époques au lieu d'une. Il dure deux fois le premier : suivez-le dans le même TensorBoard, les deux runs cochés à gauche. Puis comparez les deux dans un tableau, une ligne par run : les époques, la meilleure perte de validation et le pas où elle est atteinte, la dernière perte de validation, le meilleur checkpoint. Gardez dans `meilleur` le meilleur checkpoint des deux runs : c'est l'adaptateur que mesure l'exercice 3.

Qu'apporte la seconde époque ? Que fait la perte d'entraînement quand elle commence, et que dit alors l'écart entre la perte d'entraînement et celle de validation ? Une perte de validation plus basse fait-elle forcément un meilleur modèle ?

Une session Colab peut couper un long run. `trainer.train(resume_from_checkpoint=True)` le reprend au dernier checkpoint de `output_dir`, qui garde l'adaptateur, l'état de l'optimiseur, le planning du taux d'apprentissage et la position dans les données. Une session coupée perd aussi ses fichiers, sauf ceux de Google Drive : pour un vrai run de plusieurs heures, `output_dir` va sur Drive.

**Documentation utile**

- [Comparer des runs dans TensorBoard](https://www.tensorflow.org/tensorboard/get_started)
- [`Trainer.train`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.train)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

In [ ]:
# Le premier run fini : son état gardé, son adaptateur retiré du modèle de base.
etat_1 = trainer.state
base = trainer.model.unload()
del trainer
libere()

In [ ]:
%tensorboard --logdir sft

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `prepare("run-2", num_train_epochs=2)` : le reste de la configuration ne change pas
- Dans TensorBoard, le curseur *Smoothing* à 0 montre chaque pas tel quel ; les runs s'appellent `run-1/runs/<date>` et `run-2/runs/<date>`
- `etat.log_history` est une liste de dictionnaires : ceux qui ont une clé `"eval_loss"` sont les évaluations (avec leur `"step"`) ; `etat.num_train_epochs`, `etat.best_metric` et `etat.best_model_checkpoint` résument le run
- `pd.DataFrame(lignes, index=["run-1", "run-2"])` à partir d'une liste de dictionnaires ; `comparaison["meilleure_perte"].idxmin()` : le run de la meilleure perte

```python
trainer = prepare("run-2", num_train_epochs=...)
debut = time.time()
trainer.train()
print(f"{(time.time() - debut) / 60:.1f} min, pic {pic_go():.1f} Go")
etat_2 = trainer.state


def resume(etat) -> dict:
  evaluations = [h for h in etat.log_history if "eval_loss" in h]
  meilleure = min(evaluations, key=lambda h: h["eval_loss"])
  return {
    "epoques": etat.num_train_epochs,
    "meilleure_perte": ...,
    "pas_meilleure": meilleure["step"],
    "derniere_perte": ...,
    "checkpoint": etat.best_model_checkpoint,
  }


comparaison = pd.DataFrame([resume(etat_1), resume(etat_2)], index=["run-1", "run-2"])
print(comparaison)
meilleur = comparaison.loc[..., "checkpoint"]
print("Meilleur checkpoint :", meilleur)
```

### Solution

In [ ]:
trainer = prepare("run-2", num_train_epochs=2)
debut = time.time()
trainer.train()
print(f"{(time.time() - debut) / 60:.1f} min, pic {pic_go():.1f} Go")
etat_2 = trainer.state


def resume(etat) -> dict:
  evaluations = [h for h in etat.log_history if "eval_loss" in h]
  meilleure = min(evaluations, key=lambda h: h["eval_loss"])
  return {
    "epoques": etat.num_train_epochs,
    "meilleure_perte": meilleure["eval_loss"],
    "pas_meilleure": meilleure["step"],
    "derniere_perte": evaluations[-1]["eval_loss"],
    "checkpoint": etat.best_model_checkpoint,
  }


comparaison = pd.DataFrame([resume(etat_1), resume(etat_2)], index=["run-1", "run-2"])
print(comparaison)
meilleur = comparaison.loc[comparaison["meilleure_perte"].idxmin(), "checkpoint"]
print("Meilleur checkpoint :", meilleur)

# Au début de la seconde époque, la perte d'entraînement descend d'une marche : le
# modèle revoit des exemples qu'il a déjà appris, et l'écart avec la validation se
# creuse. Tant que la perte de validation baisse, il généralise encore ; quand elle
# remonte, il mémorise, et le meilleur checkpoint, rechargé à la fin, est celui
# d'avant. Une perte de validation plus basse ne dit pas que les réponses sont
# meilleures : l'exercice 3 mesure la tâche elle-même.

## 3. Le modèle adapté face au modèle de base

Chargez le modèle de base, `modele`, en `DTYPE`, sans quantification, et mesurez-le sur `test_part` avec la consigne seule, comme dans le TP de mesure : pour chaque ticket, `CONSIGNES` en message `system` et le message du client en message `user`, puis `genere_tout` et `evalue`. Ajoutez-lui ensuite l'adaptateur `adaptateur` (la cellule ci-dessous : votre meilleur checkpoint), fusionnez-le dans les poids, et mesurez à nouveau. L'adaptateur, appris sur le modèle en 4 bits, s'applique au modèle en 16 bits : c'est ainsi qu'on le sert. Rassemblez les deux mesures dans un tableau, avec le temps par ticket, et lisez deux ou trois réponses du modèle adapté.

Qu'a appris le modèle ? Comparez au meilleur prompt du TP de mesure, la charte et trois exemples, sans entraînement : le modèle adapté fait-il au moins aussi bien, avec un prompt bien plus court ? Qu'est-ce que ces mesures ne disent pas ?

**Documentation utile**

- [`PeftModel.from_pretrained`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.from_pretrained)
- [`merge_and_unload`](https://huggingface.co/docs/peft/package_reference/tuners#peft.tuners.tuners_utils.BaseTuner.merge_and_unload)

In [ ]:
# La mémoire des entraînements rendue au GPU.
del trainer, base
libere()
tokenizer.padding_side = "left"  # pour générer par lots

# L'adaptateur à mesurer : celui du meilleur run. Sans GPU, le vôtre n'a vu que 2
# exemples : on mesure celui que ce notebook a entraîné sur GPU avec Qwen3-0.6B.
adaptateur = meilleur
if not GPU:
  adaptateur = "support-sft-lora-qwen3-0.6b"

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)` : sans `quantization_config`, en `DTYPE`
- `PeftModel.from_pretrained(modele, adaptateur)` ajoute l'adaptateur d'un dossier (un checkpoint en est un) ; `.merge_and_unload()` l'ajoute aux poids et renvoie un modèle ordinaire, aussi rapide que celui de base
- `genere_tout` génère avec la variable `modele` : gardez ce nom au modèle adapté
- `evalue(sorties)` renvoie une `pd.Series` : ajoutez-lui le temps par ticket (`mesures["s_par_ticket"] = ...`), puis `pd.DataFrame({"base": mesures_base, "adapté": mesures_adapte})`

```python
def conversation_seule(message: str) -> list[dict]:
  return [
    {"role": "system", "content": CONSIGNES},
    {"role": "user", "content": message},
  ]


def mesure() -> tuple[list[str], pd.Series]:
  debut = time.time()
  sorties = genere_tout([conversation_seule(m) for m in test_part["message"]])
  mesures = evalue(sorties)
  mesures["s_par_ticket"] = round((time.time() - debut) / len(test_part), 1)
  return sorties, mesures


modele = AutoModelForCausalLM.from_pretrained(...)
sorties_base, mesures_base = mesure()
modele = PeftModel.from_pretrained(...).merge_and_unload()
sorties_adapte, mesures_adapte = mesure()
print(pd.DataFrame({"modèle de base": ..., "modèle adapté": ...}))
for sortie in sorties_adapte[:3]:
  print(sortie, "\n")
```

### Solution

In [ ]:
def conversation_seule(message: str) -> list[dict]:
  return [
    {"role": "system", "content": CONSIGNES},
    {"role": "user", "content": message},
  ]


def mesure() -> tuple[list[str], pd.Series]:
  debut = time.time()
  sorties = genere_tout([conversation_seule(m) for m in test_part["message"]])
  mesures = evalue(sorties)
  mesures["s_par_ticket"] = round((time.time() - debut) / len(test_part), 1)
  return sorties, mesures


modele = AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)
sorties_base, mesures_base = mesure()
modele = PeftModel.from_pretrained(modele, adaptateur).merge_and_unload()
sorties_adapte, mesures_adapte = mesure()
print(pd.DataFrame({"modèle de base": mesures_base, "modèle adapté": mesures_adapte}))
for sortie in sorties_adapte[:3]:
  print(sortie, "\n")

# Le modèle adapté a appris ce que montrent les exemples : le format, les étiquettes
# de l'équipe et les règles vérifiables de la charte, avec la seule consigne courte :
# un prompt de 150 tokens environ, contre plus de 2 000 pour la charte et ses trois
# exemples. Ces mesures ne disent
# pas si la réponse est juste (le bon délai, la bonne démarche) : c'est le travail de
# l'atelier d'évaluation.

## 4. Enregistrer et publier l'adaptateur

Un checkpoint garde de quoi reprendre l'entraînement (l'état de l'optimiseur, du planning, du générateur aléatoire) ; l'adaptateur, lui, tient en deux fichiers : `adapter_model.safetensors` (les poids) et `adapter_config.json` (sa configuration, dont le modèle de base). Copiez-les du meilleur checkpoint, `meilleur`, dans un dossier `support-sft-lora`, et comparez la taille du dossier à celle du checkpoint.

Écrivez-y aussi sa carte, `README.md` : ce qui permettra, dans six mois, de savoir d'où vient cet adaptateur. Elle commence par un en-tête YAML que lit le Hub (le modèle de base, la bibliothèque `peft`, la licence `apache-2.0`, celle de Qwen3, le jeu de données), puis donne le jeu et sa version, `<nom>/support-sft` en `v1.0` (le jeu du TP de données, dont ce notebook a chargé les fichiers), la configuration de LoRA (celle d'`adapter_config.json`), les réglages de l'entraînement du meilleur run, son meilleur checkpoint et sa perte de validation, et les versions de `trl`, `peft` et `transformers`.

Puis, si vous avez un compte Hugging Face, publiez le dossier sur le Hub, en **privé**, sous `<nom>/support-sft-lora` : l'atelier d'évaluation chargera cet adaptateur. Il faut le jeton `HF_TOKEN` du TP de données, dans les secrets de Colab (la clé dans la barre de gauche) ; sans lui, la cellule ne publie rien. Sans GPU, l'adaptateur enregistré est le vôtre, de Qwen3-0.6B, entraîné sur 2 exemples : sa carte le dit.

**Documentation utile**

- [`shutil.copy`](https://docs.python.org/3/library/shutil.html#shutil.copy)
- [Les cartes de modèle](https://huggingface.co/docs/hub/model-cards)
- [`HfApi.create_repo`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.create_repo)
- [`HfApi.upload_folder`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.upload_folder)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `shutil.copy(os.path.join(meilleur, fichier), dossier)` pour chacun des deux fichiers, après `os.makedirs(dossier, exist_ok=True)` ; `taille_mo(dossier)` (cellule du début)
- `json.load` sur `adapter_config.json` : `r`, `lora_alpha`, `lora_dropout`, `target_modules`, `base_model_name_or_path`
- Le meilleur run : `run = comparaison["meilleure_perte"].idxmin()`, puis `comparaison.loc[run]`
- Le jeton, comme au TP de données : `os.environ.get("HF_TOKEN")`, ou, dans Colab, `google.colab.userdata.get("HF_TOKEN")`, qui lève une exception quand le secret n'existe pas ; `HfApi(token=jeton).whoami()["name"]`, votre nom d'utilisateur
- `api.create_repo(depot, private=True, exist_ok=True)`, puis `api.upload_folder(folder_path=dossier, repo_id=depot)`
- Une accolade dans une f-string s'écrit `{{` ou `}}`

````python
import shutil

import peft
import transformers
import trl
from huggingface_hub import HfApi

dossier = "support-sft-lora"
os.makedirs(dossier, exist_ok=True)
for fichier in ["adapter_model.safetensors", "adapter_config.json"]:
  shutil.copy(...)
print(
  f"Adaptateur : {taille_mo(dossier):.0f} Mo ; checkpoint : {taille_mo(meilleur):.0f} Mo"
)

jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
api = HfApi(token=jeton)
nom = api.whoami()["name"] if jeton else "<nom>"

with open(os.path.join(dossier, "adapter_config.json")) as f:
  lora = json.load(f)
run = comparaison["meilleure_perte"].idxmin()
choisi = comparaison.loc[run]
carte = f"""---
base_model: {lora["base_model_name_or_path"]}
library_name: peft
license: apache-2.0
datasets:
- {nom}/support-sft
tags:
- lora
- qlora
- sft
- trl
language:
- fr
---

# support-sft-lora

Adaptateur LoRA de `{lora["base_model_name_or_path"]}` pour le service client de
Biclou & Cie (fictif) : au message d'un client, avec la consigne courte, il répond par
un objet JSON `{{"categorie", "priorite", "reponse"}}` selon la charte.

## Données

`{nom}/support-sft`, version `v1.0` : {len(entrainement)} exemples d'entraînement et
{len(validation)} de validation, au format prompt / completion de `trl`.

## Entraînement

- LoRA : rang {lora["r"]}, ...
- ...
- Meilleur checkpoint : ..., perte de validation ...
- trl {trl.__version__}, peft {...}, transformers {...}
"""
with open(os.path.join(dossier, "README.md"), "w", encoding="utf-8") as f:
  f.write(carte)
print(carte)

if jeton:
  depot = f"{nom}/support-sft-lora"
  api.create_repo(depot, private=..., exist_ok=True)
  api.upload_folder(folder_path=..., repo_id=depot)
  print("Publié :", depot)
else:
  print("Pas de jeton HF_TOKEN : l'adaptateur et sa carte restent dans", dossier)
````

### Solution

In [ ]:
import shutil

import peft
import transformers
import trl
from huggingface_hub import HfApi

dossier = "support-sft-lora"
os.makedirs(dossier, exist_ok=True)
for fichier in ["adapter_model.safetensors", "adapter_config.json"]:
  shutil.copy(os.path.join(meilleur, fichier), dossier)
print(
  f"Adaptateur : {taille_mo(dossier):.0f} Mo ; checkpoint : {taille_mo(meilleur):.0f} Mo"
)

jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
api = HfApi(token=jeton)
nom = api.whoami()["name"] if jeton else "<nom>"

with open(os.path.join(dossier, "adapter_config.json")) as f:
  lora = json.load(f)
run = comparaison["meilleure_perte"].idxmin()
choisi = comparaison.loc[run]
carte = f"""---
base_model: {lora["base_model_name_or_path"]}
library_name: peft
license: apache-2.0
datasets:
- {nom}/support-sft
tags:
- lora
- qlora
- sft
- trl
language:
- fr
---

# support-sft-lora

Adaptateur LoRA de `{lora["base_model_name_or_path"]}` pour le service client de
Biclou & Cie (fictif) : au message d'un client, avec la consigne courte, il répond par
un objet JSON `{{"categorie", "priorite", "reponse"}}` selon la charte.

## Données

`{nom}/support-sft`, version `v1.0` : {len(entrainement)} exemples d'entraînement et
{len(validation)} de validation, au format prompt / completion de `trl`.

## Entraînement

- LoRA : rang {lora["r"]}, `lora_alpha` {lora["lora_alpha"]}, dropout
  {lora["lora_dropout"]}, sur {", ".join(sorted(lora["target_modules"]))}
- Modèle de base {"en NF4, double quantification" if GPU else "en float32"}
- `SFTTrainer`, perte sur la complétion seule : {choisi["epoques"]} époque(s), taux
  {REFERENCE["learning_rate"]} en cosinus après {REFERENCE["warmup_steps"]:.0%} de
  montée, {LOT * ACCUMULATION} exemples par mise à jour, `{REFERENCE["optim"]}`,
  `max_length` {REFERENCE["max_length"]}
- Meilleur checkpoint : {run}, pas {choisi["pas_meilleure"]}, perte de validation
  {choisi["meilleure_perte"]:.3f}
- trl {trl.__version__}, peft {peft.__version__}, transformers {transformers.__version__}
"""
with open(os.path.join(dossier, "README.md"), "w", encoding="utf-8") as f:
  f.write(carte)
print(carte)

if jeton:
  depot = f"{nom}/support-sft-lora"
  api.create_repo(depot, private=True, exist_ok=True)
  api.upload_folder(folder_path=dossier, repo_id=depot)
  print("Publié :", depot)
else:
  print("Pas de jeton HF_TOKEN : l'adaptateur et sa carte restent dans", dossier)

## À retenir

- Un entraînement se suit pendant qu'il tourne : la perte d'entraînement à chaque pas, celle de validation à intervalles réguliers, le taux d'apprentissage ; TensorBoard le fait sans compte à créer, d'autres outils (Weights & Biases, MLflow, Trackio) par le même `report_to`.
- Sur un jeu *prompt / completion*, `trl` ne compte dans la perte que la réponse : le modèle apprend à répondre, pas à imiter les clients.
- Le meilleur checkpoint selon la validation, pas le dernier : `save_total_limit` borne le disque, `load_best_model_at_end` le garde et le recharge. Une seconde époque coûte autant que la première : la courbe de validation dit si elle vaut ce prix.
- Une perte de validation plus basse n'est pas un meilleur service client : on mesure la tâche, ici les règles vérifiables, et l'atelier d'évaluation y ajoutera un LLM juge.
- L'adaptateur fait quelques dizaines de Mo ; sa carte dit d'où il vient (le modèle de base, le jeu et sa version, la configuration, les versions des bibliothèques) : c'est ce qui le rend traçable, et rechargeable.